Use perceptron algorithm for detecting if SMS is spam or not

Import necessary libraries
-> Load data
-> Tokenize
-> Train and test data
-> Setup algorithm
-> Run

Import pandas and numpy

In [1]:
#Data Manipulation/Analysis
import numpy as np
import pandas as pd 

#Model Assistance
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Load Data from collection

In [2]:
# Define the path to your file

try:
    # Read the file into a pandas DataFrame

    # Assuming the file is tab-separated and has no header

    df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])

except FileNotFoundError:

    print(f"Error: The file '{'SMSSpamCollection'}' was not found. Please make sure the file is in the correct directory.")

except Exception as e:

    print(f"An error occurred: {e}")


In [3]:
#Print df to ensure the data has been loaded properly
df

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


Properly label the data

In [13]:
PreProcessedData_Label = [] #Vector for labeling message with spam or ham (1 or 0)

PreProcessedData_Message = [] #Vector with message for each label

for i, row in df.iterrows(): #Iterate through the data

    label = row['label'] #Obtain label

    message = row['message'] #Obtain message corresponding to label

    if label == 'spam': #1 is spam, 0 is ham

        PreProcessedData_Label.append(1)

    else:

        PreProcessedData_Label.append(0)

    PreProcessedData_Message.append(message.lower())  # lowercase the message

#See distribution of spam and ham
print(df['label'].value_counts())

# After the loop — transform all messages at once

#Tokenize 

vectorizer = TfidfVectorizer(
    max_features=3000,        # SMS messages are short
    stop_words='english',     
    ngram_range=(1, 2),      # Bigrams help with spam phrases
    min_df=2,                # Remove very rare words
    max_df=0.95,             # Keep more words (small dataset)
    strip_accents='ascii'    # Handle special characters
)

X = vectorizer.fit_transform(PreProcessedData_Message) 

Y = np.array(PreProcessedData_Label) 

label
ham     4825
spam     747
Name: count, dtype: int64


Split into training and testing

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.20,random_state=42)

X_train_dense = X_train.toarray()

X_test_dense = X_test.toarray()

Implementing Perceptron Algorithm

In [6]:
class Perceptron:

    #Constructor for the perceptron class
    def __init__(self, learning_rate=0.01, max_iterations=1000):

        self.learning_rate = learning_rate

        self.max_iterations = max_iterations

        self.weights = None

        self.bias = None
    
    #Training method

    def fit(self, X, Y):

        # Initialize weights and bias

        n_features = X.shape[1]

        self.weights = np.zeros(n_features)

        self.bias = 0

        
        # Training loop

        for iteration in range(self.max_iterations):

            errors = 0
            
            # Go through each training example

            for i in range(len(X)):

                # Calculate linear output

                linear_output = np.dot(X[i], self.weights) + self.bias
                
                # Make prediction (threshold at 0)

                prediction = 1 if linear_output >= 0 else 0
                
                # Update weights if prediction is wrong
                
                if prediction != Y[i]:

                    # Perceptron update rule

                    self.weights += self.learning_rate * (Y[i] - prediction) * X[i]

                    self.bias += self.learning_rate * (Y[i] - prediction)

                    errors += 1
    
    def predict(self, X):
        
        linear_output = np.dot(X, self.weights) + self.bias

        return np.where(linear_output >= 0, 1, 0)    

Training the model

In [7]:
perceptron = Perceptron(learning_rate=0.01, max_iterations=1000)

perceptron.fit(X_train_dense, Y_train)

Y_pred = perceptron.predict(X_test_dense)

accuracy = accuracy_score(Y_test, Y_pred)

print(f"\nTest Accuracy: {accuracy:.4f}")

print("Perceptron Report:")

print(classification_report(Y_test, Y_pred, target_names=['Ham', 'Spam']))

print("Error Matrix:")

cm = confusion_matrix(Y_test, Y_pred)

print(cm)


Test Accuracy: 0.9776
Perceptron Report:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       966
        Spam       0.88      0.96      0.92       149

    accuracy                           0.98      1115
   macro avg       0.94      0.97      0.95      1115
weighted avg       0.98      0.98      0.98      1115

Error Matrix:
[[947  19]
 [  6 143]]


Test our own message

In [9]:
def test_message(message, perceptron, vectorizer):
    """Test a single message for spam classification"""
    # Preprocess the message exactly like training data
    processed_message = message.lower()
    
    # Transform using the SAME vectorizer (don't fit again!)
    message_vector = vectorizer.transform([processed_message])
    message_dense = message_vector.toarray()
    
    # Make prediction
    prediction = perceptron.predict(message_dense)[0]
    
    # Get confidence (distance from decision boundary)
    linear_output = np.dot(message_dense[0], perceptron.weights) + perceptron.bias
    confidence = abs(linear_output)
    
    result = "SPAM" if prediction == 1 else "HAM"
    return result, confidence


print("\nInteractive Testing (type 'quit' to exit):")
while True:
    user_message = input("\nEnter a message to classify: ")
    if user_message.lower() == 'quit':
        break
    
    result, confidence = test_message(user_message, perceptron, vectorizer)
    print(f"Classification: {result} (confidence: {confidence:.4f})")


Interactive Testing (type 'quit' to exit):
Classification: SPAM (confidence: 0.0037)
Classification: HAM (confidence: 0.0101)
Classification: HAM (confidence: 0.0004)
Classification: HAM (confidence: 0.0063)
Classification: SPAM (confidence: 0.0002)
Classification: HAM (confidence: 0.0004)
Classification: HAM (confidence: 0.0149)
Classification: HAM (confidence: 0.0100)
Classification: HAM (confidence: 0.0083)
Classification: HAM (confidence: 0.0063)
Classification: HAM (confidence: 0.0063)
Classification: HAM (confidence: 0.0063)
